# Create dataset using preprocessed data

In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

In [2]:
from datetime import datetime

from tqdm import tqdm

# convert dates to datetime objects, and PTID and Voltage to integers
date_keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]

for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    row["Voltage"] = int(row["Voltage"])
    for key in date_keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

actual_data[0]

100%|██████████| 74063/74063 [00:00<00:00, 112034.49it/s]


{'PTID': 26053,
 'Name': 'MOUNTAIN-SWANROAD_115_104-3',
 'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
 'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
 'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
 'Voltage': 115,
 'FirstBus': 'MOUNTAIN',
 'SecondBus': 'SWANROAD',
 'OutageType': 'Planned'}

In [3]:
from pathlib import Path

output_path = Path("output")
output_path.mkdir(exist_ok=True)

# HYPER PARAMETERS

In [4]:
EVENT_WINDOW_HOURS = 6
MIN_YEAR = 2008

VOLTAGES = [69, 115, 132, 220, 345, 500, 735]
VOLTAGE_GROUP = {
    # 69 or less
    27: 69,
    34: 69,
    69: 69,
    # 155/120
    115: 115,
    120: 115,
    # 132/138
    132: 132,
    138: 132,
    # 220/230
    220: 220,
    230: 220,
    # 345
    345: 345,
    #500
    500: 500,
    # 735/765
    735: 735,
    765: 735,
}

INTERVALS_MINUTES = [15, 30, 60, 120, 240, 480]


In [5]:
outage_dates = sorted([row for row in actual_data if row["MinTimeStamp"].year >= MIN_YEAR], key=lambda x: x["MinTimeStamp"])

In [6]:
# load graph
import igraph

g = igraph.Graph.Read_Pickle("res/outage_graph.pkl")


In [7]:
from datetime import timedelta

dataset = []

delta_time = timedelta(hours=EVENT_WINDOW_HOURS)
for i, ref_row in enumerate(actual_data):
    if ref_row["MinTimeStamp"].year < MIN_YEAR:
        continue

    event_window_start = row["MinTimeStamp"] - delta_time
    window = [row for row in actual_data[:i] if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < delta_time]
    if len(window) > 10:
        break


In [8]:
window

[{'PTID': 26053,
  'Name': 'MOUNTAIN-SWANROAD_115_104-3',
  'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
  'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
  'Voltage': 115,
  'FirstBus': 'MOUNTAIN',
  'SecondBus': 'SWANROAD',
  'OutageType': 'Planned'},
 {'PTID': 26054,
  'Name': 'MOUNTAIN-SWANROAD_115_103-2',
  'OutDatetime': datetime.datetime(2005, 2, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2013, 8, 7, 8, 12, 19),
  'MaxTimeStamp': datetime.datetime(2013, 8, 7, 8, 12, 19),
  'Voltage': 115,
  'FirstBus': 'MOUNTAIN',
  'SecondBus': 'SWANROAD',
  'OutageType': 'Auto'},
 {'PTID': 25374,
  'Name': 'CHATGUAY-BEAUHARN_120_1362',
  'OutDatetime': datetime.datetime(2005, 2, 1, 0, 0),
  'MinTimeStamp': datetime.datetime(2011, 10, 19, 9, 37, 17),
  'MaxTimeStamp': datetime.datetime(2011, 10, 19, 12, 57, 17),
  'Voltage': 120,
  'FirstBus': 'CHATGUAY',
  'SecondBus': 'BEAUHARN',
  'OutageType': 'Auto'},
 {'PTID

In [9]:
def extract_features(ref_row, window):
    num_events = len(window)
    num_unique_ptids = len(set(row["PTID"] for row in window))
    voltage_group_list = {i: 0 for i in VOLTAGES}
    for row in window:
        voltage_group_list[VOLTAGE_GROUP[row["Voltage"]]] += 1
    voltage_group_list = [voltage_group_list[v] for v in VOLTAGES]
    num_planned = len([row for row in window if row["OutageType"] == "Planned"])
    num_auto = len([row for row in window if row["OutageType"] == "Auto"])
    num_unique_buses = len(set(row[bus] for row in window for bus in ["FirstBus", "SecondBus"]))

    fine_interval_features = []
    for dt in INTERVALS_MINUTES:
        dt = timedelta(minutes=dt)
        num_events_interval = len([row for row in window if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < dt])
        fine_interval_features.append(num_events_interval)

    return num_events, num_unique_ptids, voltage_group_list, num_planned, num_auto, num_unique_buses, fine_interval_features



extract_features(ref_row, window)

(14, 10, [0, 11, 3, 0, 0, 0, 0], 8, 6, 15, [14, 14, 14, 14, 14, 14])